# 상위 Generation 모델 비교 및 최종 모델 선정

Golden QA 123문항을 기준으로 상위 로컬 LLM 5종을 비교하고 최종 Generation 모델을 선정합니다.

In [ ]:
import pandas as pd
import requests
import time
import subprocess
import re
from pathlib import Path

PROJECT_DIR = Path("/mnt/c/Users/User/Desktop/[AI]스프린트미션/프로젝트 2번")
print("기본 라이브러리 임포트 완료")

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

CHROMA_PATH = str(PROJECT_DIR / "data" / "chroma_db")

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"}
)

vectorstore = Chroma(
    collection_name="rfp_docs",
    embedding_function=embedding_model,
    persist_directory=CHROMA_PATH
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print(f"Collection count: {vectorstore._collection.count()}")

In [ ]:
df_golden = pd.read_csv(str(PROJECT_DIR / "golden_dataset_v2.csv"))
print(f"골든 QA: {len(df_golden)}개")

In [ ]:
def format_money(value):
    try:
        if value in [None, "", "<unknown>"]:
            return "<unknown>"
        return f"{int(float(value)):,}원"
    except:
        return str(value)


def format_rag_context(docs):
    context_blocks = []

    for i, doc in enumerate(docs, 1):
        meta = doc.metadata

        business_amount = format_money(meta.get("사업금액", "<unknown>"))

        block = f"""
[검색결과 {i}]
[문서명] {meta.get("file_name", "<unknown>")}
[사업명] {meta.get("사업명", "<unknown>")}
[발주기관] {meta.get("발주기관", "<unknown>")}
[사업금액] {business_amount}
[입찰참여시작일] {meta.get("입찰참여시작일", "<unknown>")}
[입찰참여마감일] {meta.get("입찰참여마감일", "<unknown>")}
[섹션] {meta.get("header_path", "<unknown>")}
[문서내용]
{doc.page_content}
"""
        context_blocks.append(block.strip())

    return "\n\n".join(context_blocks)

def retrieve_multi_query(queries, k_each=3):
    all_docs = []
    seen = set()

    for q in queries:
        docs = retriever.invoke(q)

        for doc in docs[:k_each]:
            meta = doc.metadata
            key = (
                meta.get("doc_id", ""),
                meta.get("section_id", ""),
                meta.get("block_id", ""),
            )

            if key not in seen:
                all_docs.append(doc)
                seen.add(key)

    return all_docs

def is_multi_doc_question(row):
    text = str(row.get("question_type", "")) + " " + str(row.get("doc_id", "")) + " " + str(row.get("question", ""))
    
    multi_keywords = [
        "다중문서",
        "비교",
        "종합",
        "_vs_",
        "복수",
        "다른 기관",
    ]
    
    return any(keyword in text for keyword in multi_keywords)

def build_queries_for_row(row):
    question = row["question"]
    org = str(row.get("발주기관", ""))
    project = str(row.get("사업명", ""))
    doc_id = str(row.get("doc_id", ""))

    # Q010처럼 비교 대상이 명확한 경우
    if "고려대" in question or "고려대학교" in question or "광주과학기술원" in question:
        if "비교" in question:
            return [
                "고려대학교 차세대 포털 학사 정보시스템 구축사업 사업목적 사업금액 추진목표",
                "광주과학기술원 학사시스템 기능개선 사업 사업목적 사업금액 추진목표",
            ]

    # Q014처럼 교육/학습 관련 다중문서 종합
    if "교육" in question or "학습" in question or "이러닝" in question or "LMS" in question:
        if "다른 기관" in question or "없나" in question:
            return [
                "국민연금공단 2024년 이러닝시스템 운영 용역 교육 학습",
                "스포츠윤리센터 LMS 학습지원시스템 기능개선 교육 학습",
                "고려대학교 차세대 포털 학사 정보시스템 교육 학습 LMS",
                "광주과학기술원 학사시스템 기능개선 교육 학습",
            ]

    # 기본 다중문서 처리
    if is_multi_doc_question(row):
        return [
            f"{org} {project} {question}",
            question,
        ]

    # 단일문서 처리
    return [
        f"{org} {project} {question}"
    ]

1. 환경 설정

In [ ]:
UPPER_MODELS = [
    "exaone3.5:7.8b",
    "qwen2.5:7b",
    "llama3.1:8b",
    "phi4",
    "gemma3:12b",
]

In [ ]:
import requests
import time

def unload_ollama_model(model):
    """
    Ollama에서 특정 모델을 메모리/VRAM에서 unload
    """
    url = "http://127.0.0.1:11434/api/generate"

    payload = {
        "model": model,
        "prompt": "",
        "keep_alive": 0,
        "stream": False,
    }

    try:
        response = requests.post(url, json=payload, timeout=60)
        print(f"[unload] {model} status:", response.status_code)

        try:
            print(response.json())
        except Exception:
            print(response.text[:300])

    except Exception as e:
        print(f"[unload] {model} 실패:", type(e).__name__, e)

In [ ]:
import subprocess

def show_ollama_ps():
    try:
        result = subprocess.run(
            ["ollama", "ps"],
            capture_output=True,
            text=True,
            timeout=30
        )
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
    except Exception as e:
        print("ollama ps 확인 실패:", type(e).__name__, e)

In [ ]:
def get_model_options(model):
    """
    상위 모델 비교용 generation 옵션
    - 5개 모델 모두 동일 조건으로 비교
    - gemma3:12b도 1차 실행에서는 top_k를 별도로 제한하지 않음
    """
    options = {
        "temperature": 0.1,
        "num_predict": 1024,
        "top_p": 0.9,
        "repeat_penalty": 1.1,
    }

    return options

In [ ]:
def ask_ollama(model, prompt, max_retries=3, sleep_sec=2):
    url = "http://127.0.0.1:11434/api/generate"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "keep_alive": "30m",
        "options": get_model_options(model),
    }

    last_error = None

    for attempt in range(1, max_retries + 1):
        start_time = time.perf_counter()

        try:
            response = requests.post(url, json=payload, timeout=300)
            elapsed = max(round(time.perf_counter() - start_time, 2), 0)

            if response.status_code != 200:
                last_error = f"HTTP {response.status_code} 오류: {response.text}"
                print(f"[{model}] attempt {attempt} 실패:", last_error)
                time.sleep(sleep_sec)
                continue

            result = response.json()
            answer = result.get("response", "").strip()

            return {
                "model": model,
                "answer": answer,
                "elapsed_sec": elapsed,
                "attempt": attempt,
            }

        except requests.exceptions.ConnectionError:
            last_error = "Ollama 서버에 연결하지 못했습니다."
            print(f"[{model}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

        except requests.exceptions.ReadTimeout:
            last_error = "오류 발생: ReadTimeout"
            print(f"[{model}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

        except Exception as e:
            last_error = f"오류 발생: {type(e).__name__} - {e}"
            print(f"[{model}] attempt {attempt} 실패:", last_error)
            time.sleep(sleep_sec)

    return {
        "model": model,
        "answer": last_error,
        "elapsed_sec": None,
        "attempt": max_retries,
    }

In [ ]:
import pandas as pd
import time

def run_all_for_one_model(model, df_run, output_path, sleep_sec=0.5):
    results = []
    total = len(df_run)

    print("=" * 80)
    print("모델 실행 시작:", model)
    print("저장 경로:", output_path)
    print("=" * 80)

    show_ollama_ps()

    for n, (_, row) in enumerate(df_run.iterrows(), 1):
        question = row["question"]
        queries = build_queries_for_row(row)

        if len(queries) > 1:
            retrieved_docs = retrieve_multi_query(queries, k_each=3)
        else:
            retrieved_docs = retriever.invoke(queries[0])

        context = format_rag_context(retrieved_docs)

        use_multi_prompt = is_multi_doc_question(row) or len(queries) > 1

        if use_multi_prompt:
            prompt = build_multi_doc_prompt(question, context)
        else:
            prompt = build_rag_prompt(question, context)

        result = ask_ollama(model, prompt)

        results.append({
            "golden_id": row["id"],
            "golden_doc_id": row["doc_id"],
            "발주기관": row["발주기관"],
            "사업명": row["사업명"],
            "question": question,
            "golden_answer": row["answer"],
            "question_type": row["question_type"],
            "eval_metrics": row["eval_metrics"],
            "difficulty": row["difficulty"],
            "source_ref": row["source_ref"],
            "model": model,
            "model_answer": result["answer"],
            "elapsed_sec": result["elapsed_sec"],
            "attempt": result.get("attempt"),
            "queries": queries,
            "retrieved_doc_ids": [doc.metadata.get("doc_id") for doc in retrieved_docs],
            "retrieved_file_names": [doc.metadata.get("file_name") for doc in retrieved_docs],
            "retrieved_headers": [doc.metadata.get("header_path") for doc in retrieved_docs],
            "context_preview": context[:1200],
        })

        print(f"[{n}/{total}] {model} 완료:", row["id"], "|", round(result["elapsed_sec"] or -1, 2), "sec")

        time.sleep(sleep_sec)

        if n % 10 == 0:
            pd.DataFrame(results).to_csv(
                output_path,
                index=False,
                encoding="utf-8-sig"
            )

    df_result = pd.DataFrame(results)

    df_result.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("모델 실행 완료:", model, df_result.shape)

    return df_result

In [ ]:
import re

def is_short_bad_answer(x):
    x = str(x).strip()

    if len(x) >= 20:
        return False

    if x == "<unknown>":
        return False

    if re.search(r"\d", x):
        return False

    return True


def is_generation_error_answer(x):
    x = str(x).strip()

    if "Ollama 서버에 연결하지 못했습니다" in x:
        return True
    if "오류 발생" in x:
        return True
    if "HTTP" in x:
        return True
    if is_short_bad_answer(x):
        return True

    return False


def check_result_quality(df):
    error_mask = (
        df["model_answer"].astype(str).str.contains("Ollama 서버에 연결하지 못했습니다", na=False)
        | df["model_answer"].astype(str).str.contains("오류 발생", na=False)
        | df["model_answer"].astype(str).str.contains("HTTP", na=False)
        | df["elapsed_sec"].isna()
        | (df["elapsed_sec"].fillna(-1) < 0)
    )

    short_bad_mask = df["model_answer"].apply(is_short_bad_answer)

    print("전체 row:", len(df))
    print("오류 row:", error_mask.sum())
    print("짧은 비정상 답변 row:", short_bad_mask.sum())

    return df[error_mask | short_bad_mask][
        ["golden_id", "question", "model", "model_answer", "elapsed_sec"]
    ]

In [ ]:
model = "exaone3.5:7.8b"

df_exaone_78b = run_all_for_one_model(
    model=model,
    df_run=df_golden,
    output_path=str(PROJECT_DIR / "rag_golden_exaone3_5_7_8b_results.csv"),
    sleep_sec=0.8,
)

check_result_quality(df_exaone_78b)

In [ ]:
model = "qwen2.5:7b"

df_qwen_7b = run_all_for_one_model(
    model=model,
    df_run=df_golden,
    output_path=str(PROJECT_DIR / "rag_golden_qwen2_5_7b_results.csv"),
    sleep_sec=0.8,
)

check_result_quality(df_qwen_7b)

unload_ollama_model(model)
time.sleep(5)
show_ollama_ps()

In [ ]:
model = "llama3.1:8b"

df_llama_8b = run_all_for_one_model(
    model=model,
    df_run=df_golden,
    output_path=str(PROJECT_DIR / "rag_golden_llama3_1_8b_results.csv"),
    sleep_sec=0.8,
)

check_result_quality(df_llama_8b)

unload_ollama_model(model)
time.sleep(5)
show_ollama_ps()

In [ ]:
model = "phi4"

df_phi4 = run_all_for_one_model(
    model=model,
    df_run=df_golden,
    output_path=str(PROJECT_DIR / "rag_golden_phi4_results.csv"),
    sleep_sec=0.8,
)

check_result_quality(df_phi4)

unload_ollama_model(model)
time.sleep(5)
show_ollama_ps()

In [ ]:
model = "gemma3:12b"

df_gemma_12b = run_all_for_one_model(
    model=model,
    df_run=df_golden,
    output_path=str(PROJECT_DIR / "rag_golden_gemma3_12b_results.csv"),
    sleep_sec=1.0,
)

check_result_quality(df_gemma_12b)

unload_ollama_model(model)
time.sleep(5)
show_ollama_ps()

In [ ]:
df_upper_all = pd.concat(
    [
        df_exaone_78b,
        df_qwen_7b,
        df_llama_8b,
        df_phi4,
        df_gemma_12b,
    ],
    ignore_index=True,
)

df_upper_all.to_csv(
    PROJECT_DIR / "rag_golden_upper_5models_results.csv",
    index=False,
    encoding="utf-8-sig"
)

df_upper_all.shape

In [ ]:
upper_time_summary = (
    df_upper_all
    .groupby("model")["elapsed_sec"]
    .agg(["count", "mean", "median", "min", "max"])
    .reset_index()
)

upper_time_summary

In [ ]:
df_upper_all["generation_failed"] = df_upper_all["model_answer"].apply(is_generation_error_answer)

upper_failure_summary = (
    df_upper_all
    .groupby("model")["generation_failed"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

upper_failure_summary["failure_rate"] = upper_failure_summary["mean"] * 100

upper_failure_summary

In [ ]:
def show_answers_by_question(df, golden_id):
    cols = [
        "golden_id",
        "question_type",
        "question",
        "golden_answer",
        "model",
        "model_answer",
        "elapsed_sec",
        "retrieved_doc_ids",
    ]

    display(
        df[df["golden_id"] == golden_id][cols]
        .sort_values("model")
    )

In [ ]:
show_answers_by_question(df_upper_all, "Q010")  # 다중문서 비교

In [ ]:
show_answers_by_question(df_upper_all, "Q014")  # 다중문서 종합

In [ ]:
show_answers_by_question(df_upper_all, "Q042")  # 모른다 테스트

In [ ]:
show_answers_by_question(df_upper_all, "Q108")  # 가격점수 가드레일

In [ ]:
show_answers_by_question(df_upper_all, "Q111")  # 제안서/생성형

In [ ]:
important_types = [
    "단일문서_사실추출",
    "단일문서_세부요구사항",
    "다중문서_비교",
    "다중문서_종합",
    "모른다_테스트",
    "가격점수_가드레일",
    "입찰적합도_안내",
    "질문재작성",
]

df_output_review_sample = (
    df_upper_all[df_upper_all["question_type"].isin(important_types)]
    .groupby(["question_type", "model"], group_keys=False)
    .head(2)
    .copy()
)

df_output_review_sample[
    [
        "golden_id",
        "question_type",
        "question",
        "model",
        "golden_answer",
        "model_answer",
        "retrieved_doc_ids",
        "elapsed_sec",
    ]
]

In [ ]:
df_upper_failed = df_upper_all[
    df_upper_all["generation_failed"] == True
].copy()

df_upper_failed[
    [
        "golden_id",
        "question_type",
        "question",
        "model",
        "model_answer",
        "elapsed_sec",
    ]
].sort_values(["model", "golden_id"])

In [ ]:
df_upper_failed[df_upper_failed["model"] == "gemma3:12b"][
    ["golden_id", "question_type", "question", "model_answer", "elapsed_sec"]
]

In [ ]:
df_upper_all[df_upper_all["model"] == "phi4"][
    ["golden_id", "question_type", "question", "model_answer", "elapsed_sec"]
].head(20)

2. 상위 모델 5종 실행 결과

| 모델 | 평균 응답시간 | 생성 실패율 | 판정 |
|---|---|---|---|
| qwen2.5:7b | 3.89초 | 0.81% | 최종 후보 |
| exaone3.5:7.8b | 4.63초 | 0% | 최종 후보 |
| llama3.1:8b | 5.36초 | 2.44% | 참고 후보 |
| gemma3:12b | 5.41초 | 14.63% | 제외 |
| phi4 | 47.70초 | 2.44% | 제외 |

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 2000)

In [ ]:
target_ids = [
    "Q001",   # 단일문서 목적
    "Q003",   # 세부 요구사항
    "Q010",   # 다중문서 비교
    "Q014",   # 다중문서 종합
    "Q042",   # 모른다 테스트
    "Q061",   # 다중문서 비교
    "Q098",   # 예산 최대/최소
    "Q108",   # 가격점수 가드레일
    "Q111",   # 제안서 작성형
    "Q120",   # 기술점수 가드레일
]

target_models = [
    "qwen2.5:7b",
    "exaone3.5:7.8b",
]

3. 응답시간 / 실패율 비교

In [ ]:
df_manual_compare = df_upper_all[
    (df_upper_all["golden_id"].isin(target_ids)) &
    (df_upper_all["model"].isin(target_models))
].copy()

df_manual_compare = df_manual_compare.sort_values(["golden_id", "model"]).reset_index(drop=True)

df_manual_compare[
    [
        "golden_id",
        "question_type",
        "question",
        "golden_answer",
        "model",
        "model_answer",
        "retrieved_doc_ids",
        "elapsed_sec",
    ]
]

4. qwen vs exaone 수동 비교

In [ ]:
def build_rag_prompt(question, context):
    return f"""
당신은 공공 RFP 문서를 분석하는 입찰 지원 AI입니다.
아래 검색된 문서 근거만 사용하여 질문에 답변하세요.

[중요 규칙]
1. 검색된 문서 근거에 없는 내용은 추정하지 마세요.
2. 금액은 반드시 문서 또는 metadata에 있는 원문 숫자 그대로 작성하세요.
   - 억 원, 조 원, 만 원 단위로 임의 환산하지 마세요.
   - 예: 11,270,000,000원은 그대로 11,270,000,000원이라고 쓰세요.
3. 여러 문서를 비교하거나 종합하는 질문에서는 검색 결과에 포함된 발주기관과 사업명을 빠뜨리지 마세요.
4. 기술점수, 가격점수, 선정 가능성처럼 실제 평가 결과를 예측해야 하는 질문은 점수를 단정하지 마세요.
   - 평가기준과 추가로 필요한 정보를 안내하세요.
5. 전체 RFP 중 최대/최소, 평균, 순위처럼 전체 데이터 집계가 필요한 질문은 검색된 일부 문서만으로 단정하지 마세요.
   - 전체 metadata 기준 계산이 필요하다고 답변하세요.
6. 답변은 한국어 존댓말로 작성하세요.

[검색된 문서 근거]
{context}

[질문]
{question}

[답변]
"""

In [ ]:
def build_multi_doc_prompt(question, context):
    return f"""
당신은 여러 공공 RFP 문서를 비교·종합하는 입찰 지원 AI입니다.
아래 검색 결과만 근거로 사용하세요.

[다중문서 답변 규칙]
1. 검색 결과에 등장한 서로 다른 발주기관과 사업명을 먼저 확인하세요.
2. 각 발주기관별로 다음 항목을 구분해서 작성하세요.
   - 발주기관
   - 사업명
   - 사업금액
   - 주요 내용
   - 관련 근거
3. 검색 결과에 있는 기관이나 사업을 임의로 누락하지 마세요.
4. 검색 결과에 없는 기관, 사업명, 금액은 새로 만들지 마세요.
5. 금액은 원문 숫자 그대로 작성하고, 억/조 단위로 환산하지 마세요.
6. 정보가 부족하면 “검색된 근거만으로는 확인하기 어렵습니다”라고 작성하세요.
7. 답변은 한국어 존댓말로 작성하세요.

[검색된 문서 근거]
{context}

[질문]
{question}

[답변]
"""

5. 프롬프트 보완 및 재실험

전체 예산 최대/최소 질문(Q098)은 retriever 기반으로 처리 불가. metadata 집계 함수 별도 구현 필요.

> **참고:** 아래 셀들은 프롬프트 및 후처리 개선 과정을 기록한 것으로, 일부 함수가 중간 버전과 최종 버전으로 나뉘어 재정의됩니다. 최종 실행에는 마지막에 정의된 함수 버전을 사용합니다.

In [ ]:
rerun_ids = [
    "Q010",  # 금액 환산 오류
    "Q014",  # 다중문서 종합 누락/추정
    "Q098",  # 전체 예산 최대/최소
    "Q108",  # 가격점수 가드레일
    "Q120",  # 기술점수 가드레일
]

rerun_models = [
    "qwen2.5:7b",
    "exaone3.5:7.8b",
]

In [ ]:
df_rerun = df_golden[df_golden["id"].isin(rerun_ids)].copy()

df_rerun_results = []

for model in rerun_models:
    df_tmp = run_all_for_one_model(
        model=model,
        df_run=df_rerun,
        output_path=str(PROJECT_DIR / f"rerun_prompt_fixed_{model.replace(':', '_').replace('.', '_')}.csv"),
        sleep_sec=0.8,
    )
    df_rerun_results.append(df_tmp)

df_rerun_fixed = pd.concat(df_rerun_results, ignore_index=True)

df_rerun_fixed.to_csv(
    PROJECT_DIR / "rerun_prompt_fixed_qwen_exaone.csv",
    index=False,
    encoding="utf-8-sig"
)

df_rerun_fixed[
    [
        "golden_id",
        "question_type",
        "question",
        "golden_answer",
        "model",
        "model_answer",
        "retrieved_doc_ids",
        "elapsed_sec",
    ]
]

In [ ]:
# [중간 버전] 아래 함수는 프롬프트 보완 실험 과정에서 사용한 버전입니다.
# 이후 셀에서 개선된 버전으로 재정의되며, 최종 해석은 마지막 실행 결과를 기준으로 합니다.
def is_score_prediction_question(question):
    score_keywords = [
        "몇 점 받을",
        "점수 받을",
        "기술점수는 몇 점",
        "가격점수는 몇 점",
        "받을 수 있을 것 같",
        "평가점수 예상",
        "점수 예상",
    ]
    return any(keyword in question for keyword in score_keywords)

In [ ]:
# [중간 버전] 아래 함수는 프롬프트 보완 실험 과정에서 사용한 버전입니다.
# 이후 셀에서 개선된 버전으로 재정의되며, 최종 해석은 마지막 실행 결과를 기준으로 합니다.
def answer_score_prediction_guardrail(question):
    return (
        "현재 제공된 문서 근거만으로는 실제 평가점수를 예측할 수 없습니다.\n\n"
        "기술점수나 가격점수는 제안서의 구체적인 내용, 평가위원의 정성평가, "
        "입찰가격, 예정가격, 평가 산식 등 추가 정보에 따라 달라집니다.\n\n"
        "따라서 점수를 단정하기보다는, 문서에 제시된 평가 항목과 배점을 기준으로 "
        "준비해야 할 요소를 점검하는 방식으로 활용하는 것이 적절합니다."
    )

In [ ]:
if is_score_prediction_question(question):
    answer = answer_score_prediction_guardrail(question)
    result = {
        "model": model,
        "answer": answer,
        "elapsed_sec": 0.0,
        "attempt": 0,
    }
else:
    result = ask_ollama(model, prompt)

In [ ]:
# [중간 버전] 아래 함수는 프롬프트 보완 실험 과정에서 사용한 버전입니다.
# 이후 셀에서 개선된 버전으로 재정의되며, 최종 해석은 마지막 실행 결과를 기준으로 합니다.
def dedup_docs_by_doc_id(docs):
    seen = set()
    deduped = []

    for doc in docs:
        doc_id = doc.metadata.get("doc_id")
        key = doc_id if doc_id is not None else doc.page_content[:100]

        if key not in seen:
            seen.add(key)
            deduped.append(doc)

    return deduped

In [ ]:
retrieved_docs = retrieve_multi_query(queries, k_each=3)
retrieved_docs = dedup_docs_by_doc_id(retrieved_docs)
context = format_rag_context(retrieved_docs)

In [ ]:
def is_score_prediction_question(question):
    """
    기술점수/가격점수 등 실제 평가점수를 예측하려는 질문 감지
    """
    question = str(question)

    score_keywords = [
        "몇 점 받을",
        "점수 받을",
        "기술점수는 몇 점",
        "가격점수는 몇 점",
        "받을 수 있을 것 같",
        "평가점수 예상",
        "점수 예상",
        "기술점수 예상",
        "가격점수 예상",
        "몇 점 나올",
        "몇 점 가능",
    ]

    return any(keyword in question for keyword in score_keywords)

In [ ]:
def answer_score_prediction_guardrail(question):
    """
    실제 평가점수 예측을 막기 위한 rule-based 답변
    """
    return (
        "현재 제공된 문서 근거만으로는 실제 평가점수를 예측할 수 없습니다.\n\n"
        "기술점수나 가격점수는 제안서의 구체적인 작성 내용, 평가위원의 정성평가, "
        "입찰가격, 예정가격, 평가 산식, 경쟁 업체의 제안 내용 등 추가 정보에 따라 달라집니다.\n\n"
        "따라서 특정 점수를 단정하기보다는, 문서에 제시된 평가 항목과 배점을 기준으로 "
        "준비해야 할 요소를 점검하는 방식으로 활용하는 것이 적절합니다."
    )

In [ ]:
def dedup_docs_by_doc_id(docs):
    """
    검색 결과에서 같은 doc_id가 반복되는 문제를 줄이기 위한 중복 제거 함수
    """
    seen = set()
    deduped = []

    for doc in docs:
        doc_id = doc.metadata.get("doc_id")

        # doc_id가 없을 경우 chunk 앞부분으로 대체
        key = doc_id if doc_id is not None else doc.page_content[:100]

        if key not in seen:
            seen.add(key)
            deduped.append(doc)

    return deduped

In [ ]:
import pandas as pd
import time

def run_selected_for_one_model_fixed(model, df_run, output_path, sleep_sec=0.5):
    """
    문제 문항 부분 재실행용 함수
    
    적용 내용:
    1. 점수 예측형 질문은 모델 호출 전 rule-based 답변
    2. 다중 query 검색 결과는 doc_id 기준 중복 제거
    3. 기존 RAG prompt 개선 버전 사용
    """
    results = []
    total = len(df_run)

    print("=" * 80)
    print("부분 재실행 시작:", model)
    print("저장 경로:", output_path)
    print("=" * 80)

    show_ollama_ps()

    for n, (_, row) in enumerate(df_run.iterrows(), 1):
        question = row["question"]
        queries = build_queries_for_row(row)

        # 검색
        if len(queries) > 1:
            retrieved_docs = retrieve_multi_query(queries, k_each=3)
            retrieved_docs = dedup_docs_by_doc_id(retrieved_docs)
        else:
            retrieved_docs = retriever.invoke(queries[0])

        context = format_rag_context(retrieved_docs)

        use_multi_prompt = is_multi_doc_question(row) or len(queries) > 1

        if use_multi_prompt:
            prompt = build_multi_doc_prompt(question, context)
        else:
            prompt = build_rag_prompt(question, context)

        # 점수 예측형 질문은 모델 호출 전 차단
        if is_score_prediction_question(question):
            result = {
                "model": model,
                "answer": answer_score_prediction_guardrail(question),
                "elapsed_sec": 0.0,
                "attempt": 0,
            }
        else:
            result = ask_ollama(model, prompt)

        results.append({
            "golden_id": row["id"],
            "golden_doc_id": row["doc_id"],
            "발주기관": row["발주기관"],
            "사업명": row["사업명"],
            "question": question,
            "golden_answer": row["answer"],
            "question_type": row["question_type"],
            "eval_metrics": row["eval_metrics"],
            "difficulty": row["difficulty"],
            "source_ref": row["source_ref"],
            "model": model,
            "model_answer": result["answer"],
            "elapsed_sec": result["elapsed_sec"],
            "attempt": result.get("attempt"),
            "queries": queries,
            "retrieved_doc_ids": [doc.metadata.get("doc_id") for doc in retrieved_docs],
            "retrieved_file_names": [doc.metadata.get("file_name") for doc in retrieved_docs],
            "retrieved_headers": [doc.metadata.get("header_path") for doc in retrieved_docs],
            "context_preview": context[:1200],
        })

        print(
            f"[{n}/{total}] {model} 완료:",
            row["id"],
            "|",
            result["elapsed_sec"],
            "sec"
        )

        time.sleep(sleep_sec)

        if n % 5 == 0:
            pd.DataFrame(results).to_csv(
                output_path,
                index=False,
                encoding="utf-8-sig"
            )

    df_result = pd.DataFrame(results)

    df_result.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("부분 재실행 완료:", model, df_result.shape)

    return df_result

In [ ]:
def run_selected_model_then_unload_fixed(model, df_run, output_path, sleep_sec=0.8):
    """
    문제 문항 부분 재실행 후 모델 unload까지 수행
    """
    df_result = run_selected_for_one_model_fixed(
        model=model,
        df_run=df_run,
        output_path=output_path,
        sleep_sec=sleep_sec,
    )

    print("\n실행 직후 ollama ps:")
    show_ollama_ps()

    unload_ollama_model(model)

    time.sleep(5)

    print("\nunload 이후 ollama ps:")
    show_ollama_ps()

    return df_result

In [ ]:
rerun_ids = [
    "Q010",  # 금액 표기 / 다중문서 비교
    "Q014",  # 다중문서 종합
    "Q098",  # 전체 예산 최대/최소
    "Q108",  # 가격점수 계산 불가
    "Q120",  # 기술점수 예측 방지
]

df_rerun = df_golden[df_golden["id"].isin(rerun_ids)].copy()

df_rerun[["id", "question_type", "question", "answer"]]

In [ ]:
df_qwen_fixed = run_selected_model_then_unload_fixed(
    model="qwen2.5:7b",
    df_run=df_rerun,
    output_path=str(PROJECT_DIR / "rerun_fixed_qwen2_5_7b.csv"),
    sleep_sec=0.8,
)

In [ ]:
df_exaone_fixed = run_selected_model_then_unload_fixed(
    model="exaone3.5:7.8b",
    df_run=df_rerun,
    output_path=str(PROJECT_DIR / "rerun_fixed_exaone3_5_7_8b.csv"),
    sleep_sec=0.8,
)

In [ ]:
df_rerun_fixed = pd.concat(
    [df_qwen_fixed, df_exaone_fixed],
    ignore_index=True
)

df_rerun_fixed.to_csv(
    PROJECT_DIR / "rerun_fixed_qwen_exaone.csv",
    index=False,
    encoding="utf-8-sig"
)

df_rerun_fixed[
    [
        "golden_id",
        "question_type",
        "question",
        "golden_answer",
        "model",
        "model_answer",
        "retrieved_doc_ids",
        "elapsed_sec",
    ]
]

In [ ]:
target_models = ["qwen2.5:7b", "exaone3.5:7.8b"]

df_candidate_all = df_upper_all[
    df_upper_all["model"].isin(target_models)
].copy()

df_candidate_all.shape

In [ ]:
import re

def has_foreign_language_mix(text):
    text = str(text)
    # 중국어/일본어 문자 포함 여부
    return bool(re.search(r"[\u4e00-\u9fff\u3040-\u30ff]", text))

def has_money_conversion_risk(text):
    text = str(text)
    # 원문 금액 뒤에 억/조/만 원 환산이 섞인 경우를 의심
    return bool(re.search(r"\d[\d,]*원.*(억|조|만)", text))

def is_too_short_answer(text):
    text = str(text).strip()
    if text == "<unknown>":
        return False
    return len(text) < 20

def has_score_prediction(text):
    text = str(text)
    patterns = [
        r"\d+\s*점",
        r"\d+\s*~\s*\d+\s*점",
        r"받을 수 있습니다",
        r"예상할 수",
    ]
    return any(re.search(p, text) for p in patterns)

df_candidate_all["too_short"] = df_candidate_all["model_answer"].apply(is_too_short_answer)
df_candidate_all["foreign_mix"] = df_candidate_all["model_answer"].apply(has_foreign_language_mix)
df_candidate_all["money_conversion_risk"] = df_candidate_all["model_answer"].apply(has_money_conversion_risk)
df_candidate_all["score_prediction_risk"] = df_candidate_all["model_answer"].apply(has_score_prediction)

df_candidate_all[
    [
        "model",
        "too_short",
        "foreign_mix",
        "money_conversion_risk",
        "score_prediction_risk",
    ]
].sum()

6. 자동 점검 및 수동 평가

### 자동 점검 결과 (플래그 건수)

| 항목 | exaone3.5:7.8b | qwen2.5:7b |
|---|---|---|
| 짧은 출력 | 0 | 3 |
| 외국어 혼입 | 0 | 5 |
| 금액 환산 위험 | 3 | 1 |
| 점수 예측 위험 | 0 | 2 |
| 생성 실패 | 0 | 1 |
| **플래그 합계** | **3** | **12** |

### 수동 평가 결과 (평균 점수)

| 평가 항목 | exaone3.5:7.8b | qwen2.5:7b |
|---|---|---|
| 검색 적합성 (retrieval_ok) | 1.000 | 0.909 |
| 답변 정확성 (answer_correct) | 0.600 | 0.409 |
| 근거성 (grounded) | 0.800 | 0.773 |
| 환각 방지 (no_hallucination) | 0.800 | 0.727 |
| 형식 준수 (format_followed) | 0.700 | 0.545 |
| metadata 활용 (metadata_used) | 0.900 | 0.864 |
| **전체 평균** | **0.800** | **0.705** |


In [ ]:
df_suspicious = df_candidate_all[
    df_candidate_all[
        [
            "too_short",
            "foreign_mix",
            "money_conversion_risk",
            "score_prediction_risk",
            "generation_failed",
        ]
    ].any(axis=1)
].copy()

df_suspicious[
    [
        "golden_id",
        "question_type",
        "question",
        "model",
        "model_answer",
        "too_short",
        "foreign_mix",
        "money_conversion_risk",
        "score_prediction_risk",
        "elapsed_sec",
    ]
].sort_values(["golden_id", "model"])

In [ ]:
df_type_sample = (
    df_candidate_all
    .sort_values(["question_type", "golden_id", "model"])
    .groupby(["question_type", "model"], group_keys=False)
    .head(2)
    .copy()
)

df_type_sample[
    [
        "golden_id",
        "question_type",
        "question",
        "golden_answer",
        "model",
        "model_answer",
        "retrieved_doc_ids",
        "elapsed_sec",
    ]
]

In [ ]:
df_manual_review = pd.concat(
    [df_suspicious, df_type_sample],
    ignore_index=True
).drop_duplicates(
    subset=["golden_id", "model"]
).sort_values(["golden_id", "model"])

eval_cols = [
    "retrieval_ok",
    "answer_correct",
    "grounded",
    "no_hallucination",
    "format_followed",
    "metadata_used",
    "memo",
]

for col in eval_cols:
    if col not in df_manual_review.columns:
        df_manual_review[col] = ""

df_manual_review.to_csv(
    PROJECT_DIR / "qwen_exaone_manual_review_expanded.csv",
    index=False,
    encoding="utf-8-sig"
)

df_manual_review[
    [
        "golden_id",
        "question_type",
        "question",
        "golden_answer",
        "model",
        "model_answer",
        "retrieved_doc_ids",
        "elapsed_sec",
    ] + eval_cols
]

In [ ]:
import pandas as pd
import re

# ── 데이터 로드 ──
df_qwen = pd.read_csv(PROJECT_DIR / "rag_golden_qwen2_5_7b_results.csv")
df_exaone = pd.read_csv(PROJECT_DIR / "rag_golden_exaone3_5_7_8b_results.csv")
df_check = pd.concat([df_qwen, df_exaone], ignore_index=True)

print(f"전체 row 수: {len(df_check)} (qwen {len(df_qwen)} + exaone {len(df_exaone)})")

In [ ]:
# [중간 버전] 아래 함수는 프롬프트 보완 실험 과정에서 사용한 버전입니다.
# 이후 셀에서 개선된 버전으로 재정의되며, 최종 해석은 마지막 실행 결과를 기준으로 합니다.
# ── 자동 점검 함수 ──

def check_too_short(x, min_len=15):
    x = str(x).strip()
    if x in ["<unknown>", "nan", ""]:
        return False
    return len(x) < min_len

def check_foreign_mix(x):
    x = str(x)
    # 중국어, 일본어, 키릴 문자 감지
    patterns = [
        r'[\u4e00-\u9fff]',   # 중국어
        r'[\u3040-\u30ff]',   # 일본어
        r'[\u0400-\u04ff]',   # 키릴
    ]
    return any(re.search(p, x) for p in patterns)

def check_money_conversion_risk(x):
    x = str(x)
    # 억/조/만 원 환산 표현 감지
    patterns = [
        r'\d+억\s*원',
        r'\d+조\s*원',
        r'약\s*\d+억',
        r'약\s*\d+조',
        r'\(\s*약\s*\d+',
    ]
    return any(re.search(p, x) for p in patterns)

def check_score_prediction_risk(x):
    x = str(x)
    keywords = [
        "점 받을 수 있",
        "점을 받을",
        "점이 예상",
        "점 정도 받",
        "점수를 받을 수 있을",
    ]
    return any(kw in x for kw in keywords)

def check_generation_failed(x):
    x = str(x).strip()
    if len(x) < 10:
        return True
    error_keywords = ["Ollama 서버에 연결", "오류 발생", "HTTP Error", "ReadTimeout"]
    return any(kw in x for kw in error_keywords)

def check_unknown_overuse(x):
    x = str(x)
    count = x.count("<unknown>")
    return count >= 2

# ── 점검 실행 ──
df_check["too_short"]              = df_check["model_answer"].apply(check_too_short)
df_check["foreign_mix"]            = df_check["model_answer"].apply(check_foreign_mix)
df_check["money_conversion_risk"]  = df_check["model_answer"].apply(check_money_conversion_risk)
df_check["score_prediction_risk"]  = df_check["model_answer"].apply(check_score_prediction_risk)
df_check["generation_failed"]      = df_check["model_answer"].apply(check_generation_failed)
df_check["unknown_overuse"]        = df_check["model_answer"].apply(check_unknown_overuse)

flag_cols = [
    "too_short",
    "foreign_mix",
    "money_conversion_risk",
    "score_prediction_risk",
    "generation_failed",
    "unknown_overuse",
]

df_check["any_flag"] = df_check[flag_cols].any(axis=1)

print(f"전체 row: {len(df_check)}")
print(f"플래그 있는 row: {df_check['any_flag'].sum()}")

In [ ]:
# ── 모델별 플래그 요약 ──
flag_summary = (
    df_check
    .groupby("model")[flag_cols + ["any_flag"]]
    .sum()
    .astype(int)
)
flag_summary["전체_문항"] = 123
flag_summary

In [ ]:
# ── 플래그된 row 상세 확인 ──
df_flagged = df_check[df_check["any_flag"] == True].copy()

df_flagged[[
    "golden_id",
    "question_type",
    "question",
    "model",
    "model_answer",
    "elapsed_sec",
] + flag_cols].sort_values(["model", "golden_id"])

In [ ]:
# ── 저장 ──
df_check.to_csv(
    PROJECT_DIR / "qwen_exaone_auto_check_results.csv",
    index=False,
    encoding="utf-8-sig"
)
df_flagged.to_csv(
    PROJECT_DIR / "qwen_exaone_flagged_rows.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"저장 완료: qwen_exaone_auto_check_results.csv ({len(df_check)}행)")
print(f"저장 완료: qwen_exaone_flagged_rows.csv ({len(df_flagged)}행)")

In [ ]:
import pandas as pd
import re

# ── 1. 데이터 로드 및 플래그 재생성 ──
df_qwen = pd.read_csv(PROJECT_DIR / "rag_golden_qwen2_5_7b_results.csv")
df_exaone = pd.read_csv(PROJECT_DIR / "rag_golden_exaone3_5_7_8b_results.csv")
df_check = pd.concat([df_qwen, df_exaone], ignore_index=True)

def check_too_short(x, min_len=15):
    x = str(x).strip()
    if x in ["<unknown>", "nan", ""]:
        return False
    return len(x) < min_len

def check_foreign_mix(x):
    patterns = [r'[\u4e00-\u9fff]', r'[\u3040-\u30ff]', r'[\u0400-\u04ff]']
    return any(re.search(p, str(x)) for p in patterns)

def check_money_conversion_risk(x):
    patterns = [r'\d+억\s*원', r'\d+조\s*원', r'약\s*\d+억', r'약\s*\d+조', r'\(\s*약\s*\d+']
    return any(re.search(p, str(x)) for p in patterns)

def check_score_prediction_risk(x):
    keywords = ["점 받을 수 있", "점을 받을", "점이 예상", "점 정도 받", "점수를 받을 수 있을"]
    return any(kw in str(x) for kw in keywords)

def check_generation_failed(x):
    x = str(x).strip()
    if len(x) < 10:
        return True
    return any(kw in x for kw in ["Ollama 서버에 연결", "오류 발생", "HTTP Error", "ReadTimeout"])

flag_cols = ["too_short", "foreign_mix", "money_conversion_risk",
             "score_prediction_risk", "generation_failed"]

df_check["too_short"]             = df_check["model_answer"].apply(check_too_short)
df_check["foreign_mix"]           = df_check["model_answer"].apply(check_foreign_mix)
df_check["money_conversion_risk"] = df_check["model_answer"].apply(check_money_conversion_risk)
df_check["score_prediction_risk"] = df_check["model_answer"].apply(check_score_prediction_risk)
df_check["generation_failed"]     = df_check["model_answer"].apply(check_generation_failed)
df_check["any_flag"]              = df_check[flag_cols].any(axis=1)

df_flagged = df_check[df_check["any_flag"] == True].copy()
print(f"플래그 row: {len(df_flagged)}개")
print(df_flagged.groupby("model")["any_flag"].count())

In [ ]:
# ── 2. question_type별 정상 샘플 추출 (모델별 2개씩) ──
df_normal = df_check[df_check["any_flag"] == False].copy()
df_normal = df_normal.reset_index(drop=True)

normal_parts = []
for (qtype, model), group in df_normal.groupby(["question_type", "model"]):
    sampled = group.sample(min(2, len(group)), random_state=42)
    normal_parts.append(sampled)

normal_sample = pd.concat(normal_parts, ignore_index=True)

print(f"정상 샘플 수: {len(normal_sample)}")
for (qtype, model), g in normal_sample.groupby(["question_type", "model"]):
    print(f"  {qtype} / {model}: {len(g)}개")

In [ ]:
# ── 3. 플래그 row + 정상 샘플 합치기 ──
eval_cols = [
    "golden_id", "question_type", "question",
    "golden_answer", "model", "model_answer",
    "elapsed_sec", "retrieved_doc_ids",
] + flag_cols

df_manual_pool = pd.concat(
    [df_flagged[eval_cols], normal_sample[eval_cols]],
    ignore_index=True
).drop_duplicates(subset=["golden_id", "model"]).sort_values(
    ["question_type", "golden_id", "model"]
).reset_index(drop=True)

# 수동 평가 컬럼 추가
for col in ["retrieval_ok", "answer_correct", "grounded",
            "no_hallucination", "format_followed", "metadata_used", "memo"]:
    df_manual_pool[col] = ""

print(f"수동평가 대상 전체: {len(df_manual_pool)}행")
print(df_manual_pool.groupby(["question_type", "model"]).size().to_string())

In [ ]:
# ── 4. 저장 ──
df_manual_pool.to_csv(
    PROJECT_DIR / "qwen_exaone_manual_eval_pool.csv",
    index=False,
    encoding="utf-8-sig"
)
print("저장 완료: qwen_exaone_manual_eval_pool.csv")

In [ ]:
# 수동평가 우선순위 문항 출력
priority_ids = [
    "Q001",  # 단일문서 사실추출
    "Q003",  # 세부 요구사항
    "Q010",  # 다중문서 비교 (금액 환산 문제)
    "Q014",  # 다중문서 종합 (누락/환각 문제)
    "Q039",  # generation_failed (qwen)
    "Q042",  # 모른다 테스트
    "Q052",  # too_short (qwen) - 실제 정답인지 확인
    "Q077",  # foreign_mix (qwen)
    "Q097",  # 모른다 + foreign_mix (qwen)
    "Q108",  # 가격점수 가드레일
    "Q114",  # 입찰적합도 + foreign_mix
    "Q116",  # 질문재작성 (양쪽 플래그)
    "Q120",  # 기술점수 가드레일 미적용
]

df_priority = df_manual_pool[
    df_manual_pool["golden_id"].isin(priority_ids)
].sort_values(["golden_id", "model"])

eval_display_cols = [
    "golden_id", "question_type", "question",
    "golden_answer", "model", "model_answer",
] + flag_cols

pd.set_option("display.max_colwidth", 300)

for _, row in df_priority.iterrows():
    print("=" * 80)
    print(f"[{row['golden_id']}] {row['question_type']} | {row['model']}")
    print(f"질문: {row['question']}")
    print(f"정답: {str(row['golden_answer'])[:200]}")
    print(f"답변: {str(row['model_answer'])[:300]}")
    flags = [col for col in flag_cols if row[col]]
    print(f"플래그: {flags if flags else '없음'}")
    print()

In [ ]:
# ── 수동평가 결과 입력 ──
manual_scores = {
    # (golden_id, model): {항목: 점수}
    ("Q010", "exaone3.5:7.8b"): {
        "retrieval_ok": "O", "answer_correct": "△", "grounded": "O",
        "no_hallucination": "O", "format_followed": "X", "metadata_used": "O",
        "memo": "금액 환산값 붙임(약 1조 1천억 원)"
    },
    ("Q010", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "△", "grounded": "O",
        "no_hallucination": "O", "format_followed": "O", "metadata_used": "O",
        "memo": "금액 원문 표기 준수, 일부 세부내용 누락"
    },
    ("Q014", "exaone3.5:7.8b"): {
        "retrieval_ok": "O", "answer_correct": "X", "grounded": "X",
        "no_hallucination": "X", "format_followed": "O", "metadata_used": "△",
        "memo": "검색 근거 없는 기관 생성, 사업명 오기재"
    },
    ("Q014", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "X", "grounded": "X",
        "no_hallucination": "X", "format_followed": "O", "metadata_used": "△",
        "memo": "사업금액 추정값 생성(3.2억 추정)"
    },
    ("Q039", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "△", "grounded": "O",
        "no_hallucination": "O", "format_followed": "O", "metadata_used": "O",
        "memo": "배경 4가지 중 1가지만 답변, generation_failed는 오탐"
    },
    ("Q042", "exaone3.5:7.8b"): {
        "retrieval_ok": "O", "answer_correct": "O", "grounded": "O",
        "no_hallucination": "O", "format_followed": "△", "metadata_used": "O",
        "memo": "[문서 근거 부족] 내부 라벨 형태 노출"
    },
    ("Q042", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "O", "grounded": "O",
        "no_hallucination": "O", "format_followed": "O", "metadata_used": "O",
        "memo": "깔끔하게 확인 불가 답변"
    },
    ("Q052", "qwen2.5:7b"): {
        "retrieval_ok": "△", "answer_correct": "X", "grounded": "△",
        "no_hallucination": "X", "format_followed": "O", "metadata_used": "△",
        "memo": "정답 14억원인데 4억으로 오답"
    },
    ("Q077", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "△", "grounded": "O",
        "no_hallucination": "O", "format_followed": "X", "metadata_used": "O",
        "memo": "답변 중 중국어 혼입"
    },
    ("Q097", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "O", "grounded": "O",
        "no_hallucination": "O", "format_followed": "X", "metadata_used": "O",
        "memo": "마지막 문장 중국어 혼입"
    },
    ("Q108", "exaone3.5:7.8b"): {
        "retrieval_ok": "O", "answer_correct": "△", "grounded": "O",
        "no_hallucination": "O", "format_followed": "O", "metadata_used": "O",
        "memo": "정보 부족 판단은 맞으나 필요 정보 구체적 언급 없음"
    },
    ("Q108", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "△", "grounded": "O",
        "no_hallucination": "O", "format_followed": "O", "metadata_used": "O",
        "memo": "정보 부족 판단은 맞으나 필요 정보 구체적 언급 없음"
    },
    ("Q114", "qwen2.5:7b"): {
        "retrieval_ok": "△", "answer_correct": "X", "grounded": "X",
        "no_hallucination": "X", "format_followed": "X", "metadata_used": "△",
        "memo": "유사 사업 추정 생성, 사업금액 0원, 외국어 혼입"
    },
    ("Q116", "exaone3.5:7.8b"): {
        "retrieval_ok": "O", "answer_correct": "O", "grounded": "O",
        "no_hallucination": "O", "format_followed": "O", "metadata_used": "O",
        "memo": "회사 정보 없이 RFP 기준 요건 나열, 범위 내 답변"
    },
    ("Q116", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "△", "grounded": "O",
        "no_hallucination": "O", "format_followed": "X", "metadata_used": "O",
        "memo": "내용 중 중국어 혼입"
    },
    ("Q120", "qwen2.5:7b"): {
        "retrieval_ok": "O", "answer_correct": "X", "grounded": "O",
        "no_hallucination": "O", "format_followed": "X", "metadata_used": "O",
        "memo": "가드레일 미적용, 배점 최대치를 받을 수 있는 점수로 오답"
    },
}

# ── df_manual_pool에 반영 ──
score_map = {"O": 1.0, "△": 0.5, "X": 0.0}
eval_cols = ["retrieval_ok", "answer_correct", "grounded",
             "no_hallucination", "format_followed", "metadata_used"]

for (gid, model), scores in manual_scores.items():
    mask = (df_manual_pool["golden_id"] == gid) & (df_manual_pool["model"] == model)
    for col, val in scores.items():
        df_manual_pool.loc[mask, col] = val

# 점수 컬럼 추가
for col in eval_cols:
    df_manual_pool[f"{col}_score"] = df_manual_pool[col].map(score_map)

df_manual_pool["total_score"] = df_manual_pool[[f"{c}_score" for c in eval_cols]].mean(axis=1)

print("수동평가 입력 완료")
print(f"평가 완료 row: {df_manual_pool['answer_correct'].ne('').sum()}개")

In [ ]:
# ── 모델별 점수 집계 ──
scored = df_manual_pool[df_manual_pool["answer_correct"].ne("")].copy()

score_cols = [f"{c}_score" for c in eval_cols]
summary = scored.groupby("model")[score_cols + ["total_score"]].mean().round(3)
summary.columns = eval_cols + ["total_avg"]
print("=== 모델별 평균 점수 ===")
print(summary.to_string())

print()
print("=== question_type별 total_score ===")
print(scored.groupby(["question_type", "model"])["total_score"].mean().round(3).to_string())

In [ ]:
# ── 저장 ──
df_manual_pool.to_csv(
    PROJECT_DIR / "qwen_exaone_manual_eval_scored.csv",
    index=False,
    encoding="utf-8-sig"
)
print("저장 완료: qwen_exaone_manual_eval_scored.csv")

In [ ]:
improved_rag_qa_prompt_v2 = """
너는 공공 RFP 문서를 분석하는 AI입니다.
반드시 아래 [검색된 문서 내용]만 근거로 사용자의 질문에 답변하세요.

핵심 규칙:
1. 문서에 있는 내용만 사용하세요.
2. 문서에 없는 내용은 절대 추측하거나 생성하지 마세요.
3. 금액은 반드시 원문 숫자 그대로만 표기하세요.
   - 예: 11,270,000,000원
   - 절대 금지: 약 112억 7천만 원, (약 1조 1천억 원), 약 15억 등
   - 괄호로 환산값을 추가하는 것도 금지입니다.
4. 기간, 날짜는 원문 표기를 그대로 유지하세요.
5. 목록형 요구사항은 항목을 누락하지 말고 모두 포함하세요.
6. 문서에 근거가 부족하면 부족하다고 말하고, 필요한 정보를 구체적으로 안내하세요.
7. 반드시 한국어 존댓말로만 작성하세요.
   - 영어, 중국어, 일본어 등 한국어 외 언어를 절대 사용하지 마세요.
   - 한 글자라도 외국어가 섞이면 안 됩니다.
8. 내부 처리 라벨(예: [문서 근거 부족], [검색결과 N], 판단:, 출력: 등)을 답변에 노출하지 마세요.
9. 검색된 문서 목록에 없는 기관, 사업명, 금액은 절대 생성하지 마세요.
10. 질문에서 묻지 않은 정보는 답변에 포함하지 마세요.

[검색된 문서 내용]
{context}

[사용자 질문]
{question}

[답변]
""".strip()

print("improved_rag_qa_prompt_v2 정의 완료")

In [ ]:
improved_multi_doc_prompt_v2 = """
너는 공공 RFP 문서를 분석하는 AI입니다.
아래 [검색된 문서 내용]에 포함된 여러 문서를 종합하여 질문에 답변하세요.

핵심 규칙:
1. 검색된 문서 목록에 포함된 발주기관과 사업만 사용하세요.
   - 검색 결과에 없는 기관이나 사업을 절대 추가하지 마세요.
2. 각 문서는 최대 1회만 언급하세요. 같은 사업을 반복하지 마세요.
3. 금액은 반드시 원문 숫자 그대로만 표기하세요.
   - 절대 금지: 약 112억, (약 1조 원), 약 15억 등 환산 표현
4. 반드시 한국어 존댓말로만 작성하세요.
   - 영어, 중국어, 일본어 등 한국어 외 언어를 절대 사용하지 마세요.
5. 답변은 발주기관 / 사업명 / 사업금액 / 주요 내용 순으로 정리하세요.
6. 문서에 없는 내용은 추측하지 말고 확인 불가로 표기하세요.
7. 내부 처리 라벨을 답변에 노출하지 마세요.

[검색된 문서 내용]
{context}

[사용자 질문]
{question}

[답변]
""".strip()

print("improved_multi_doc_prompt_v2 정의 완료")

In [ ]:
# [중간 버전] 아래 함수는 프롬프트 보완 실험 과정에서 사용한 버전입니다.
# 이후 셀에서 개선된 버전으로 재정의되며, 최종 해석은 마지막 실행 결과를 기준으로 합니다.
import re

def postprocess_v3(text: str, model_name: str = "") -> dict:
    result = {
        "original": text,
        "processed": text,
        "flags": [],
        "blocked": False
    }

    foreign_patterns = {
        "중국어": r'[\u4e00-\u9fff]',
        "일본어": r'[\u3040-\u30ff]',
        "키릴":  r'[\u0400-\u04ff]',
    }
    detected = [lang for lang, pat in foreign_patterns.items()
                if re.search(pat, text)]
    if detected:
        result["flags"].append(f"foreign_mix:{','.join(detected)}")
        lines = text.split("\n")
        clean_lines = [
            line for line in lines
            if not any(re.search(pat, line) for pat in foreign_patterns.values())
        ]
        result["processed"] = "\n".join(clean_lines).strip()
        if not result["processed"]:
            result["processed"] = "죄송합니다. 답변을 생성하는 중 오류가 발생했습니다. 다시 질문해 주세요."
            result["blocked"] = True

    money_patterns = [
        r'\d+억\s*원', r'\d+조\s*원',
        r'약\s*\d+억', r'약\s*\d+조',
        r'\(\s*약\s*\d+',
    ]
    if any(re.search(p, result["processed"]) for p in money_patterns):
        result["flags"].append("money_conversion_risk")

    label_patterns = [
        r'\[문서 근거 부족\]',
        r'\[검색결과\s*\d+\]',
        r'판단\s*:\s*',
        r'출력\s*:\s*',
    ]
    for pat in label_patterns:
        result["processed"] = re.sub(pat, "", result["processed"]).strip()

    if not result["processed"].strip():
        result["processed"] = "죄송합니다. 답변을 생성하는 중 오류가 발생했습니다. 다시 질문해 주세요."
        result["flags"].append("empty_response")
        result["blocked"] = True

    return result

print("postprocess_v3 정의 완료")

In [ ]:
# [중간 버전] 아래 함수는 prompt v2 재실험에 사용한 버전입니다. 이후 전체 Golden QA 재실행 단계에서 최종 버전으로 재정의됩니다.
def ask_with_rag_v2(question, retriever, model_name,
                    is_multi_doc=False, max_retries=2):

    if is_score_prediction_question(question):
        return {
            "model_answer": answer_score_prediction_guardrail(question),
            "elapsed_sec": 0.0,
            "attempt": 0,
            "queries": [],
            "retrieved_doc_ids": [],
            "post_flags": [],
            "guardrail_applied": "score_prediction"
        }

    if is_multi_doc:
        docs = retrieve_multi_query([question])
    else:
        docs = retriever.invoke(question)

    context = format_rag_context(docs)

    if is_multi_doc:
        prompt = improved_multi_doc_prompt_v2.format(
            context=context, question=question
        )
    else:
        prompt = improved_rag_qa_prompt_v2.format(
            context=context, question=question
        )

    result = ask_ollama(model_name, prompt)

    answer = result.get("answer", "") or ""
    elapsed = result.get("elapsed_sec") or 0.0
    attempt = result.get("attempt", 0)

    post = postprocess_v3(answer, model_name)

    return {
        "model_answer": post["processed"],
        "elapsed_sec": elapsed,
        "attempt": attempt,
        "queries": [question],
        "retrieved_doc_ids": [d.metadata.get("doc_id", "") for d in docs],
        "post_flags": post["flags"],
        "guardrail_applied": None
    }

print("ask_with_rag_v2 정의 완료")

In [ ]:
rerun_targets = [
    {"golden_id": "Q010", "question": "고려대학교 차세대 포털 사업과 광주과학기술원 학사시스템 사업을 비교해줘", "is_multi_doc": True},
    {"golden_id": "Q014", "question": "교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?", "is_multi_doc": True},
    {"golden_id": "Q077", "question": "사업 목적과 주요 구축 범위는?", "is_multi_doc": False},
    {"golden_id": "Q097", "question": "버스 노선 몇 개를 커버하는가?", "is_multi_doc": False},
    {"golden_id": "Q116", "question": "우리 회사가 이 입찰에 적합한지 봐줘.", "is_multi_doc": False},
    {"golden_id": "Q120", "question": "기술점수는 몇 점 받을 수 있을 것 같아?", "is_multi_doc": False},
]

rerun_models = ["qwen2.5:7b", "exaone3.5:7.8b"]
rerun_results = []

for model_name in rerun_models:
    print(f"\n=== {model_name} 재실험 시작 ===")
    for target in rerun_targets:
        print(f"  {target['golden_id']} 처리 중...")
        res = ask_with_rag_v2(
            question=target["question"],
            retriever=retriever,
            model_name=model_name,
            is_multi_doc=target["is_multi_doc"]
        )
        rerun_results.append({
            "golden_id": target["golden_id"],
            "question": target["question"],
            "model": model_name,
            **res
        })
    unload_ollama_model(model_name)
    print(f"{model_name} 완료 및 언로드")

df_rerun = pd.DataFrame(rerun_results)
print(f"\n재실험 완료: {len(df_rerun)}행")

In [ ]:
for _, row in df_rerun.iterrows():
    print("=" * 70)
    print(f"[{row['golden_id']}] {row['model']}")
    print(f"질문: {row['question']}")
    if row.get("guardrail_applied"):
        print(f"[가드레일 적용: {row['guardrail_applied']}]")
    print(f"답변: {str(row['model_answer'])[:400]}")
    if row.get("post_flags"):
        print(f"후처리 플래그: {row['post_flags']}")
    print()

In [ ]:
df_rerun.to_csv(
    PROJECT_DIR / "rerun_prompt_v2_results.csv",
    index=False,
    encoding="utf-8-sig"
)
print("저장 완료: rerun_prompt_v2_results.csv")

In [ ]:
def build_multi_queries(question: str) -> list:
    """
    다중문서 비교/종합 질문에서 비교 대상을 추출해 쿼리 리스트 반환.
    추출 실패 시 원본 질문을 단일 쿼리로 반환.
    """
    # 패턴 1: A와 B를 비교 / A랑 B 비교
    pattern_compare = re.findall(
        r'([\w가-힣\s·]+(?:사업|시스템|프로젝트|용역))',
        question
    )
    if len(pattern_compare) >= 2:
        return [q.strip() for q in pattern_compare[:3]]

    # 패턴 2: 기관명 추출
    pattern_org = re.findall(
        r'([\w가-힣]+(?:대학교|기술원|공단|센터|청|원|공사|연구원))',
        question
    )
    if len(pattern_org) >= 2:
        return [f"{org} 사업" for org in pattern_org[:3]]

    return [question]

# 테스트
test_q = "고려대학교 차세대 포털 사업과 광주과학기술원 학사시스템 사업을 비교해줘"
print(build_multi_queries(test_q))

test_q2 = "교육이나 학습 관련해서 다른 기관이 발주한 사업은 없나?"
print(build_multi_queries(test_q2))

In [ ]:
def postprocess_v3(text: str, model_name: str = "") -> dict:
    result = {
        "original": text,
        "processed": text,
        "flags": [],
        "blocked": False,
        "need_retry": False
    }

    foreign_patterns = {
        "중국어": r'[\u4e00-\u9fff]',
        "일본어": r'[\u3040-\u30ff]',
        "키릴":  r'[\u0400-\u04ff]',
    }
    detected = [lang for lang, pat in foreign_patterns.items()
                if re.search(pat, text)]
    if detected:
        result["flags"].append(f"foreign_mix:{','.join(detected)}")
        lines = text.split("\n")
        clean_lines = [
            line for line in lines
            if not any(re.search(pat, line) for pat in foreign_patterns.values())
        ]
        cleaned = "\n".join(clean_lines).strip()

        # 제거 후 너무 짧으면 재시도 요청
        if len(cleaned) < 30:
            result["need_retry"] = True
            result["processed"] = text  # 원본 유지하고 재시도
        else:
            result["processed"] = cleaned

    money_patterns = [
        r'\d+억\s*원', r'\d+조\s*원',
        r'약\s*\d+억', r'약\s*\d+조',
        r'\(\s*약\s*\d+',
    ]
    if any(re.search(p, result["processed"]) for p in money_patterns):
        result["flags"].append("money_conversion_risk")

    label_patterns = [
        r'\[문서 근거 부족\]',
        r'\[검색결과\s*\d+\]',
        r'판단\s*:\s*',
        r'출력\s*:\s*',
    ]
    for pat in label_patterns:
        result["processed"] = re.sub(pat, "", result["processed"]).strip()

    if not result["processed"].strip():
        result["processed"] = "죄송합니다. 답변을 생성하는 중 오류가 발생했습니다. 다시 질문해 주세요."
        result["flags"].append("empty_response")
        result["blocked"] = True

    return result

print("postprocess_v3 수정 완료")

In [ ]:
# [중간 버전] 아래 함수는 prompt v2 재실험에 사용한 버전입니다.
# 이후 전체 Golden QA 재실행 단계에서는 row 기반 query 보정이 추가된 최종 버전으로 재정의됩니다.
def ask_with_rag_v2(question, retriever, model_name,
                    is_multi_doc=False, max_retries=2):

    if is_score_prediction_question(question):
        return {
            "model_answer": answer_score_prediction_guardrail(question),
            "elapsed_sec": 0.0,
            "attempt": 0,
            "queries": [],
            "retrieved_doc_ids": [],
            "post_flags": [],
            "guardrail_applied": "score_prediction"
        }

    if is_multi_doc:
        queries = build_multi_queries(question)
        docs = retrieve_multi_query(queries)
    else:
        docs = retriever.invoke(question)

    context = format_rag_context(docs)

    if is_multi_doc:
        prompt = improved_multi_doc_prompt_v2.format(
            context=context, question=question
        )
    else:
        prompt = improved_rag_qa_prompt_v2.format(
            context=context, question=question
        )

    answer = ""
    elapsed = 0.0
    attempt = 0

    for attempt in range(max_retries + 1):
        result = ask_ollama(model_name, prompt)
        answer = result.get("answer", "") or ""
        elapsed = result.get("elapsed_sec") or 0.0

        post = postprocess_v3(answer, model_name)

        # 외국어 제거 후 내용이 충분하면 종료
        if not post["need_retry"] and len(post["processed"].strip()) >= 10:
            break

        print(f"  [{model_name}] attempt {attempt+1} 재시도 (foreign_mix 또는 빈 응답)")

    return {
        "model_answer": post["processed"],
        "elapsed_sec": elapsed,
        "attempt": attempt,
        "queries": queries if is_multi_doc else [question],
        "retrieved_doc_ids": [d.metadata.get("doc_id", "") for d in docs],
        "post_flags": post["flags"],
        "guardrail_applied": None
    }

print("ask_with_rag_v2 수정 완료")

In [ ]:
rerun_results = []

for model_name in rerun_models:
    print(f"\n=== {model_name} 재실험 시작 ===")
    for target in rerun_targets:
        print(f"  {target['golden_id']} 처리 중...")
        res = ask_with_rag_v2(
            question=target["question"],
            retriever=retriever,
            model_name=model_name,
            is_multi_doc=target["is_multi_doc"]
        )
        rerun_results.append({
            "golden_id": target["golden_id"],
            "question": target["question"],
            "model": model_name,
            **res
        })
    unload_ollama_model(model_name)
    print(f"{model_name} 완료 및 언로드")

df_rerun = pd.DataFrame(rerun_results)
print(f"\n재실험 완료: {len(df_rerun)}행")

In [ ]:
for _, row in df_rerun.iterrows():
    print("=" * 70)
    print(f"[{row['golden_id']}] {row['model']}")
    print(f"질문: {row['question']}")
    if row.get("guardrail_applied") and row["guardrail_applied"] != "nan":
        print(f"[가드레일 적용: {row['guardrail_applied']}]")
    print(f"쿼리: {row.get('queries', [])}")
    print(f"검색 문서: {row.get('retrieved_doc_ids', [])}")
    print(f"답변: {str(row['model_answer'])[:400]}")
    if row.get("post_flags"):
        print(f"후처리 플래그: {row['post_flags']}")
    print()

df_rerun.to_csv(
    PROJECT_DIR / "rerun_prompt_v2_results.csv",
    index=False,
    encoding="utf-8-sig"
)
print("저장 완료: rerun_prompt_v2_results.csv")

In [ ]:
def ask_with_rag_v2(question, retriever, model_name,
                    is_multi_doc=False, max_retries=2,
                    row=None):

    if is_score_prediction_question(question):
        return {
            "model_answer": answer_score_prediction_guardrail(question),
            "elapsed_sec": 0.0,
            "attempt": 0,
            "queries": [],
            "retrieved_doc_ids": [],
            "post_flags": [],
            "guardrail_applied": "score_prediction"
        }

    if row is not None:
        queries = build_queries_for_row(row)
    elif is_multi_doc:
        queries = build_multi_queries(question)
    else:
        queries = [question]

    docs = retrieve_multi_query(queries)
    context = format_rag_context(docs)

    if is_multi_doc:
        prompt = improved_multi_doc_prompt_v2.format(
            context=context, question=question
        )
    else:
        prompt = improved_rag_qa_prompt_v2.format(
            context=context, question=question
        )

    answer = ""
    elapsed = 0.0
    attempt = 0

    for attempt in range(max_retries + 1):
        result = ask_ollama(model_name, prompt)
        answer = result.get("answer", "") or ""
        elapsed = result.get("elapsed_sec") or 0.0

        post = postprocess_v3(answer, model_name)

        if not post["need_retry"] and len(post["processed"].strip()) >= 10:
            break

        print(f"  [{model_name}] attempt {attempt+1} 재시도")

    return {
        "model_answer": post["processed"],
        "elapsed_sec": elapsed,
        "attempt": attempt,
        "queries": queries,
        "retrieved_doc_ids": [d.metadata.get("doc_id", "") for d in docs],
        "post_flags": post["flags"],
        "guardrail_applied": None
    }

print("ask_with_rag_v2 최종 수정 완료")

In [ ]:
MULTI_DOC_TYPES = {"다중문서_비교", "다중문서_종합"}
print("MULTI_DOC_TYPES 정의 완료")

In [ ]:
df_golden = pd.read_csv(PROJECT_DIR / "golden_dataset_v2.csv")
print(f"골든 QA 총 {len(df_golden)}개")

target_models = ["qwen2.5:7b", "exaone3.5:7.8b"]
all_results = []

for model_name in target_models:
    print(f"\n=== {model_name} 시작 ===")
    for idx, row in df_golden.iterrows():
        golden_id = row.get("golden_id") or row.get("id") or f"Q{idx+1:03d}"
        question = str(row.get("question", ""))
        question_type = str(row.get("question_type", ""))
        is_multi = question_type in MULTI_DOC_TYPES

        print(f"  [{golden_id}] {question_type} 처리 중...")

        res = ask_with_rag_v2(
            question=question,
            retriever=retriever,
            model_name=model_name,
            is_multi_doc=is_multi,
            row=row
        )

        all_results.append({
            "golden_id": golden_id,
            "golden_doc_id": row.get("golden_doc_id", ""),
            "발주기관": row.get("발주기관", ""),
            "사업명": row.get("사업명", ""),
            "question": question,
            "golden_answer": row.get("golden_answer", ""),
            "question_type": question_type,
            "difficulty": row.get("difficulty", ""),
            "model": model_name,
            **res
        })

    unload_ollama_model(model_name)
    print(f"{model_name} 완료 및 언로드")

df_v2_all = pd.DataFrame(all_results)
print(f"\n전체 실행 완료: {len(df_v2_all)}행")

In [ ]:
df_v2_all.to_csv(
    PROJECT_DIR / "rag_golden_v2_prompt_all_results.csv",
    index=False,
    encoding="utf-8-sig"
)
print("저장 완료: rag_golden_v2_prompt_all_results.csv")

7. 최종 모델 선정

**최종 모델: `exaone3.5:7.8b`**

| 기준 | 내용 |
|---|---|
| 생성 실패율 | 0% |
| 외국어 혼입 | 0건 |
| 수동평가 평균 | 0.800 |
| 응답시간 | 평균 4.63초 |